In [34]:
!pip install pandas
!pip install transformers
!pip install kagglehub
!pip install torch


In [35]:

from pathlib import Path
import pandas as pd
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import kagglehub


In [36]:
#@title If this cell crashes after download just re-run
from google.colab import files
from google.colab import drive

model_name = kagglehub.model_download("qwen-lm/qwen-3/transformers/0.6b-fp8")
!wget https://raw.githubusercontent.com/abelardo-p/CS-175-Codebase/refs/heads/main/colab_files/test_dataset.csv
!wget https://raw.githubusercontent.com/abelardo-p/CS-175-Codebase/refs/heads/main/colab_files/db_schemas_with_complexity.json


--2026-03-16 20:39:27--  https://raw.githubusercontent.com/abelardo-p/CS-175-Codebase/refs/heads/main/colab_files/test_dataset.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 287637 (281K) [text/plain]
Saving to: ‘test_dataset.csv’

test_dataset.csv    100%[===================>] 280.90K  --.-KB/s    in 0.004s  

2026-03-16 20:39:27 (74.1 MB/s) - ‘test_dataset.csv’ saved [287637/287637]

--2026-03-16 20:39:27--  https://raw.githubusercontent.com/abelardo-p/CS-175-Codebase/refs/heads/main/colab_files/db_schemas_with_complexity.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTT

In [37]:
PREFIX = "```sql\n"
SUFFIX = "```"

In [38]:
def formatPrompt(system_prompt: str, assumption, schema: str, question: str) -> str:
    return [f"{system_prompt}\n\nSchema:\n{schema}\n\n{assumption}\n\n", \
            f"Question: {question}"]

def formatOutput(prefix: str, suffix: str, query_result) -> str:
    if query_result.startswith(prefix) and query_result.endswith(suffix):
        return query_result.split(prefix)[1].split(suffix)[0]
    elif query_result.startswith(prefix) and query_result.endswith(suffix + '\n'):
        return query_result.split(prefix)[1].split(suffix)[0]
    else:
        return query_result

In [39]:
class Pipeline:
    def __init__(self, model: str, system_prompt: str, assumption: str, data_path: Path, data_scheme_path):
        self.model_name = model
        self.system_prompt = system_prompt
        self.assumption = assumption
        self.data_scheme_path = data_scheme_path
        self.data_path = data_path

        self.tokenizer = AutoTokenizer.from_pretrained(model)
        self.model = AutoModelForCausalLM.from_pretrained(
          model_name,
          torch_dtype="auto",
          device_map="auto"
        )

    def promptLLM(self, prompt: list[str]) -> str:

      # load the tokenizer and the model
      # prepare the model input
      messages = [
          {"role": "system", "content": prompt[0]},
          {"role": "user", "content": prompt[1]}

      ]
      text = self.tokenizer.apply_chat_template(
          messages,
          tokenize=False,
          add_generation_prompt=True,
          enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
      )
      model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

      # conduct text completion
      generated_ids = self.model.generate(
          **model_inputs,
          max_new_tokens=1024
      )
      output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
      try:
        # rindex finding 151668 ()
        index = len(output_ids) - output_ids[::-1].index(151668)
      except ValueError:
        index = 0

      thinking_content = self.tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
      content = self.tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
      return content

    def run(self):
        with open(self.data_scheme_path, "r") as f:
            db_schemas = json.load(f)
        df = pd.read_csv(self.data_path)

        result_map = {
            "db_id": [],
            "question": [],
            "query": [],
            "pred_query": [],
            }

        prompt = None
        for i in range(len(df)):
            curr_row = df.iloc[i]
            db_id = curr_row['db_id']
            question = curr_row['question']
            query = curr_row['query']
            db = db_schemas[db_id]
            schema = db['ddl_string'].lower()

            prompt = formatPrompt(self.system_prompt, self.assumption, schema, question)

            predicted_sql = self.promptLLM(prompt)
            predicted_sql = formatOutput(PREFIX, SUFFIX, predicted_sql)

            result_map['db_id'].append(db_id)
            result_map['question'].append(question)
            result_map['query'].append(query)
            result_map['pred_query'].append(predicted_sql)

            if i < 10:
                print("Question: ", question)
                print("Gold Query: ", query)
                print("Predicted SQL: ", predicted_sql)
                print()
            else:
              break
        result_df = pd.DataFrame(result_map)
        result_df.to_csv(f"predictions-qwen3:0.6b:fp8.csv", index=False)
        files.download("predictions-qwen3:0.6b:fp8.csv")
        # !cp /content/predictions-qwen3:32b:fp8.csv /content/drive/MyDrive/

In [40]:
def main():
    data_path = Path('/content/test_dataset.csv')
    data_scheme_path = Path('/content/db_schemas_with_complexity.json')

    system_prompt = 'You are a text-to-SQL assistant. Generate a correct SQL query that adheres to the given assumptions, using only the question and database schema, and nothing else. Do not include any additional non SQL content such as comments or formatting'
    assumption = "# Assumptions for generating the SQL query:\n# 1. Only table names are aliased\n# 2. LIMIT value is always a numerical type\n# 3. Only one of INTERSECT / UNION / EXCEPT can be used"

    return Pipeline(model_name, system_prompt, assumption, data_path, data_scheme_path)

In [41]:
pipeline = main()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [42]:
# @title Stop anytime once results are visible
pipeline.run()

Question:  How many singers do we have?
Gold Query:  SELECT count(*) FROM singer
Predicted SQL:  SELECT COUNT(*) FROM singer;

Question:  What is the total number of singers?
Gold Query:  SELECT count(*) FROM singer
Predicted SQL:  SELECT COUNT(*) FROM singer;

Question:  Show name, country, age for all singers ordered by age from the oldest to the youngest.
Gold Query:  SELECT name ,  country ,  age FROM singer ORDER BY age DESC
Predicted SQL:  SELECT name, country, age FROM singer ORDER BY age;

Question:  What are the names, countries, and ages for every singer in descending order of age?
Gold Query:  SELECT name ,  country ,  age FROM singer ORDER BY age DESC
Predicted SQL:  SELECT name, country, age FROM singer ORDER BY age DESC;

Question:  What is the average, minimum, and maximum age of all singers from France?
Gold Query:  SELECT avg(age) ,  min(age) ,  max(age) FROM singer WHERE country  =  'France'
Predicted SQL:  SELECT AVG(age), MIN(age), MAX(age) FROM singer WHERE country

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The execution accuracy and semantic similarity stage of the pipeline are too large and involved to make work in a colab notebook given the time